# Covered Today:
1. Open Source Models for Code Generation
2. Building a Gradio UI to Test Python-to-C++ Code Conversion Models
3. Qwen 3 Coder vs GPT OSS: OpenRouter Model Performance Showdown

as step 1, we start by looking for the best open source models.

On artificial analysis we have:
1. Terminal Bench
    1. GLM-5.3 (max)
    2. GLM-5.3-Flash
    3. DeepSeek V4.1 Flash (max)
    4. DeepSeek V4 Pro 0813 (max)
    5. Kimi K3 (max)
2. Sci Code
    1. Kimi K3 (max)
    2. GLM-5.3 (max)
    3. DeepSeek V4.1 Flash (max)
    4. GLM-5.3-Flash
    5. DeepSeek V4 Pro 0813 (max)

Then on vellum we have
1. Live Code Bench
    1. DeepSeek V4 Pro
    2. DeepSeek V4 Flash
    3. Kimi K2 Thinking
    4. GPT oss 120b
    5. GPT oss 20b
2. SWE Bench
    1. DeepSeek V4 Pro
    2. MiniMax M3
    3. DeepSeek V4 Flash
    4. Kimi K2 Thinking
    5. DeepSeek-R1

Next we move to seal
1. SWE Atlas
    1. GLM 5.2 (Mini-SWE-Agent)
    2. Kimi-K2.5 (Mini-SWE-Agent)
    3. Minimax-M2.5 (Mini-SWE-Agent)

Next up is live bench, here we have
1. Kimi K3
2. DeepSeek V4.1 Flash Max Effort
3. GLM-5.2
4. GLM-5.3
5. GLM-5.3 Flash

Finally we have arena and here under coding, best open source models are
1. Kimi K3
2. Hy4 preview
3. GLM 5.2
4. DeepSeek V4 Pro
5. DeepSeek V4.1 Flash

Now basis the above data, the model count is as follows:

01. DeepSeek V4 Pro - 5
02. Kimi K3 - 4
03. DeepSeek V4.1 Flash - 4
04. GLM-5.3 - 3
05. GLM-5.3 Flash - 3
06. GLM 5.2 - 3
07. DeepSeek V4 Flash - 2
08. Kimi K2 Thinking - 2
09. DeepSeek-R1 - 1
10. GPT oss 120b - 1
11. GPT oss 20b - 1
12. Hy4 preview - 1
13. Kimi-K2.5 - 1
14. MiniMax M3 - 1
15. Minimax-M2.5 - 1

Basis the above data, our 5 top models are below. Also, as the next step, look at their model charts, to understand if these can be run on our system or not. We are looking for parameters less than 8B(which becomes 16gigs.)

01. DeepSeek V4 Pro - 1.6T params - $0.5795 / $1.738per 1M
02. Kimi K3 - 2.8T params - $1.875 / $10.50per 1M
03. DeepSeek V4.1 Flash - 763B params - $0.15 / $0.60per 1M
04. GLM-5.3 - 753B params - $0.8775 / $2.97per 1M
05. GLM-5.3 Flash - 321B params - $0.075 / $0.25per 1M

To this list, we also add GPT oss 20B($0.02 / $0.10per 1M) and 120B($0.03 / $0.17per 1M) as I have heard good things about these. We will use open-router to run all these agents.

In [6]:
# since we will be using open router, lets start by loading the required keys and the libraries. 
import os
from dotenv import load_dotenv
import gradio as gr 
from openai import OpenAI

In [4]:
# now, load open router key
load_dotenv(override=True)
open_router_key = os.getenv("OPENROUTER_API_KEY")

# lets add the base URL
open_router_url = "https://openrouter.ai/api/v1"

In [5]:
# we know that the key is loaded and has the correct format, so we now move ahead to checking if we are getting thr response from the API
user_prompt = "This is a test to check if the system is connected to the LLM. If you receive this message, and are able to reply, reply example with: The system is working. Model Name: {your name}."
messages = [{"role": "user", "content": user_prompt}]
open_router_client = OpenAI(api_key=open_router_key, base_url=open_router_url)
response = open_router_client.chat.completions.create(model='nvidia/nemotron-3-ultra-550b-a55b:free', messages=messages)
print(response.choices[0].message.content)

The system is working. Model Name: Nemotron 3 Ultra.


In [7]:
# Now, since we have to build this in gradio, our functions must be designed according to that also. First let's finalise the app design.
# 1. row one to have two parts, one for showing the existing code, and second to show the code that gets ported. Basically, the python code on the left and converted c++ code on the right.
# 2. Then we will have another row, where, we will have the buttons and drop downs: 1. to run python code, 2. to port python code to c++ (for now, we might add more language options later), 3. To run c++ code and finally and finally we will have a drop down to select from the given LLM options.
# 3. Again, we will have one row, with two rows, where, we will have 1. the output of the python code and 2. the output of the c++ code. 

# We can also store the results in sqlite, but this is something we can work on later, since this would need us to create a new page with the stored data. For now, we work on the 3 rows. 

# Now, for these 3 rows to work as is, we need the following:
# 1. A function, that returns the python code as the output. > For now, this shall remain static and loaded by default.
# 2. A function, that runs this python code and returns the output. > Run Python button runs this function and its output is loaded to python output section.
# 3. A function, that ports the given python code and returns the C++ code. > port python runs this > and its output is displayed in the show c++ code. In the background, it will also save the code to the file. For this, the input will be the model name.
# 4. A function to compile and run the c++ code. The input of this will be the compiled file. Output will be displayed in the c++ output section. 